In [5]:
import pandas as pd

# Load directly from URL - no kaggle needed!
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"

columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
           'Insulin', 'BMI', 'DiabetesPedigree', 'Age', 'Outcome']

df = pd.read_csv(url, names=columns)

print('Dataset shape:', df.shape)
print(df.head())

Dataset shape: (768, 9)
   Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0            6      148             72             35        0  33.6   
1            1       85             66             29        0  26.6   
2            8      183             64              0        0  23.3   
3            1       89             66             23       94  28.1   
4            0      137             40             35      168  43.1   

   DiabetesPedigree  Age  Outcome  
0             0.627   50        1  
1             0.351   31        0  
2             0.672   32        1  
3             0.167   21        0  
4             2.288   33        1  


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

# Step 1 - Split features and target
X = df.drop('Outcome', axis=1)
y = df['Outcome']

# Step 2 - Split into train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Step 3 - Scale the data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Step 4 - Train 3 models and compare
models = {
    'Logistic Regression': LogisticRegression(),
    'Random Forest': RandomForestClassifier(n_estimators=100),
    'SVM': SVC(kernel='rbf')
}

print("Model Comparison:")
print("-" * 35)
for name, model in models.items():
    model.fit(X_train, y_train)
    acc = accuracy_score(y_test, model.predict(X_test))
    print(f"{name}: {acc*100:.2f}%")

# Step 5 - Detailed report for best model (Random Forest)
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
print("\nDetailed Report (Random Forest):")
print(classification_report(y_test, y_pred,
      target_names=['No Diabetes', 'Diabetes']))

Model Comparison:
-----------------------------------
Logistic Regression: 75.32%
Random Forest: 74.03%
SVM: 73.38%

Detailed Report (Random Forest):
              precision    recall  f1-score   support

 No Diabetes       0.79      0.78      0.78        99
    Diabetes       0.61      0.62      0.61        55

    accuracy                           0.72       154
   macro avg       0.70      0.70      0.70       154
weighted avg       0.72      0.72      0.72       154



In [7]:
import joblib

# Save the best model (Logistic Regression won!)
joblib.dump(scaler, '/content/scaler.pkl')
joblib.dump(models['Logistic Regression'], '/content/disease_model.pkl')
print("Model saved!")

Model saved!


In [8]:
from google.colab import files
files.download('/content/disease_model.pkl')
files.download('/content/scaler.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
from xgboost import XGBClassifier

xgb = XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss')
xgb.fit(X_train, y_train)
acc = accuracy_score(y_test, xgb.predict(X_test))
print(f"XGBoost: {acc*100:.2f}%")

XGBoost: 72.08%
